In [1]:
# Cell 1: Load
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from src.data_processor import RetailDataProcessor

raw_path = ROOT / 'data' / 'raw' / 'online_retail.xlsx'
processor = RetailDataProcessor(raw_path)
df_raw = processor.load_and_explore()

In [2]:
# Cell 2: Clean
processor.clean_data()
processor.engineer_features()
print('Valid rows:', len(processor.df_valid))


=== BÁO CÁO AUDIT LÀM SẠCH DỮ LIỆU ===
1. Số dòng ban đầu:                 541,909
2. Số dòng trùng lặp (Duplicate):   5,268 (Chênh lệch Qty: 13,948, Rev: £21,740.98)
3. Số dòng sau khi xử lý Duplicate: 536,641
4. Lỗi thiếu/không hợp lệ trường cốt lõi: 0
5. Số dòng hóa đơn hủy (Prefix C):  9,251
6. Lỗi Quantity/UnitPrice <= 0:     2,512
7. Giao dịch bán hợp lệ giữ lại:    524,878 (Trong đó thiếu CustomerID: 132,186)


-> Tổng Revenue hợp lệ sau làm sạch: £10,642,110.80
Valid rows: 524878


In [3]:
# Cell 3: Analyze
result = processor.analyze_revenue()
print('Total revenue:', round(result['total'], 2))
print(result['by_country'].head(5))

C:\Users\LENOVO\.matplotlib is not a writable directory


Matplotlib created a temporary cache directory at C:\Users\LENOVO\AppData\Local\Temp\matplotlib-ottf0b32 because there was an issue with the default path (C:\Users\LENOVO\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Total revenue: 10642110.8
Country
United Kingdom    9001744.094
Netherlands        285446.340
EIRE               283140.520
Germany            228678.400
France             209625.370
Name: Revenue, dtype: float64


In [4]:
# Cell 4: Export
proc_dir = ROOT / 'data' / 'processed'
processor.export_data(
    proc_dir / 'cleaned_retail.csv',
    proc_dir / 'cancelled_retail.csv',
)
print('Exported.')

Exported.


In [5]:
# list: top 5 sản phẩm theo revenue (list of tuple + sort + comprehension)
prod_rev = processor.df_valid.groupby('StockCode')['Revenue'].sum()
top5 = sorted(prod_rev.items(), key=lambda kv: kv[1], reverse=True)[:5]
top5_codes = [code for code, rev in top5]
print('Top 5 stock codes by revenue:', top5_codes)

Top 5 stock codes by revenue: ['DOT', 22423, 23843, '85123A', 47566]


In [6]:
# dict: đếm đơn theo quốc gia (.get với default)
country_counts = {}
for c in processor.df_valid['Country']:
    country_counts[c] = country_counts.get(c, 0) + 1
print('Total country entries recorded:', len(country_counts))

Total country entries recorded:

 38


In [7]:
# set: quốc gia duy nhất + phép toán tập hợp
all_countries = set(processor.df_valid['Country'].unique())
print('United Kingdom in all_countries:', 'United Kingdom' in all_countries)
print('Unique countries count:', len(all_countries))

United Kingdom in all_countries: True
Unique countries count: 38


In [8]:
# tuple: unpacking + làm key của dict
code, revenue = top5[0]
rev_lookup = {(code, 'United Kingdom'): revenue}
print('Sample tuple lookup key:', (code, 'United Kingdom'), '-> Revenue:', rev_lookup[(code, 'United Kingdom')])

Sample tuple lookup key: ('DOT', 'United Kingdom') -> Revenue: 206248.77


In [9]:
# Demo các hàm business rules độc lập từ src.business_rules
from src.business_rules import (
    calculate_revenue,
    is_cancelled_invoice,
    order_month,
    assign_customer_segment,
)

print('Sample revenue calculation (Qty=5, Price=12.5):', calculate_revenue(5, 12.5))
print('Is cancelled (C536365):', is_cancelled_invoice('C536365'))
print('Order month (2011-12-01 08:26:00):', order_month('2011-12-01 08:26:00'))
print('Assigned segment (R=5, F=5, M=5):', assign_customer_segment(5, 5, 5))

Sample revenue calculation (Qty=5, Price=12.5): 62.5
Is cancelled (C536365): True
Order month (2011-12-01 08:26:00): 2011-12
Assigned segment (R=5, F=5, M=5): Champions
